In [ ]:
import scipy.stats as sps
import numpy as np
import matplotlib.pyplot as plt
import maintkit.distributions as rd
import maintkit.probability_plotting as rpp
import maintkit.wiener as rw
import maintkit.imperfect_maintenance as rim
# Imported directly rather than as a module: maintkit exports a CLASS named
# poisson_process, which shadows the module of the same name, so
# `import maintkit.poisson_process as rp` binds the class instead.
from maintkit.poisson_process import poisson_process, power_law_nhpp

# Example 1
*The lifetime of a certain device is exponentially distributed with a mean value of 500 hours. What is the probability of this device operating for 600 hours?*

**Solution.** 
The failure times $T$ satisfy
$$ T\sim\mathrm{Exp}\left(\lambda\right) $$
where "$\sim$" can be read as "distributed as". The lone parameter is $\lambda$, which can be computed as
$$\lambda = \frac{1}{MTTF}$$
and the probability of operating for 600 hours is simply the reliability
$$R(600) = 1-F(600)$$
We can use the reliability module (with scipy.stats) to evaluate this:

In [ ]:
pd = rd.expdist()
pd = pd(scale=500)
R1 = pd.reliability(600)
print("The probability of surviving 600 hours is {:.1f}%".format(R1*100))

# Example 2
*Significant rail defects in a particular corridor occur at an average rate of one every three months. Answer the following:*

*1. What is the probability of two failures occurring less than one month apart?*  
*2. What is the probability that there are more than 5 failures in a year?*

**Solution.** Let $X$ denote the time between failure events. Assuming that the average rate is constant implies that $X\sim \mathrm{Exp}(\lambda)$ with $\lambda=\frac{1}{3}$ failures per month. This can be seen by noting that the number of failures in a time interval $k$ follows a Poisson distribution when the average arrival rate is constant. Thus, the probability that k=0 events occur in time $\Delta t$ is
\begin{align*}
\mathbb{P}\left[ k=0 \right] & = 1-\mathbb{P}\left[ k\geq 0 \right] \\
                             & = \frac{(\lambda \Delta t)^k e^{-\lambda \Delta t}}{k!} \\
                             & = \frac{(\lambda \Delta t)^0 e^{-\lambda \Delta t}}{0!} \\
                             & = e^{-\lambda \Delta t} \\
\end{align*}
The probability that two failures occur less than one month apart is equivalent to the inter-failure time being less than one month, i.e.
\begin{align*}
\mathbb{P}\left[ k\geq1 \right] & = 1-e^{-\frac{1}{3}\cdot 1} \\
\end{align*}
which is just the CDF of the exponential distribution. Using scipy.stats:

In [ ]:
pd1 = rd.expdist()(scale=3.0)
answer1 = pd1.cdf(1.0) # one month
print("The probability of two failures occurring within a month of each other is {:.2f}%".format(answer1*100))

For 2, we note that $k\sim\mathrm{Poisson}(\lambda\Delta t)$ and we can evaluate this using scipy.stats as

In [ ]:
pd2 = sps.poisson(mu=1/3*12)
answer2 = 1-pd2.cdf(5)
print("The probability of more than 5 failures occurring within a year is {:.2f}%".format(answer2*100))

# Example 3
*A machine model you want to acquire has a Weibull reliability profile with $\beta=0.8$ and $\eta=30$ years. Any failure of the machine would cost 200,000. On the market you find:*  
1. *a new machine for 2M and*  
2. *a 5-year-old one for 1.8M*

*Which one do you buy?*

**Solution.**
We begin by noting that the reliability of the new machine is
\begin{align*}
    R(t) & = \exp\left[-\left(\frac{t}{30} \right)^{0.8} \right]
\end{align*}
while the reliability of the five-year-old machine is
\begin{align*}
    R(t+5\mid 5) & = \exp\left[-\left(\frac{t+5}{30} \right)^{0.8} + \left(\frac{5}{30} \right)^{0.8} \right]
\end{align*}
It's not immediately obvious which reliability is better, so we plot them. First, let's define the conditional reliability distribution as a custom distribution

In [ ]:
eta = 30
beta = 0.8
t0 = 50
pd = rd.weibull()
pd = pd(beta,scale=eta)
MTTF = pd.mean()

plt.rcParams.update({'font.size': 16})
fig,axs = plt.subplots(nrows=2,ncols=1,figsize=(15,7))
pd.plot(type='reliability',ax=axs[0],pltkwds={'color':'blue','label':'R(t)'})
pd.plot(type='conditional_reliability',ax=axs[0],t0=t0,pltkwds={'color':'red','ls':'--','label':'R(t|t0)'})
axs[0].set_xlabel('Age')
axs[0].legend()

pd.plot(type='hazard',ax=axs[1],pltkwds={'color':'blue','label':'Hazard'})
axs[1].set_xlabel('Age')
axs[1].set_ylabel('$\lambda(t)$')
axs[1].legend()


We can see that the older asset has higher reliabilty due to its suvival of a portion of its infant mortality period. Thus, we should purchase the five-year-old one.

# Example 4
Empirical CDF with complete data using various methods.

In [ ]:
ti = [128.04, 93.12, 67.8, 68.64, 84.12, 17.88, 105.84, 28.92, 33, 41.52, 42.12, 68.88, 45.6, 48.4, 51.84, 51.96, 68.64, 54.12, 98.64, 127.92, 55.56, 105.12, 173.40]
ti = np.array(ti)
observed = np.ones(ti.shape)

t1,Fhat1 = rpp.ecdf(ti,observed,plot=False)
t2,Fhat2 = rpp.ecdf(ti,observed,pos="mean",plot=False)
t3,Fhat3 = rpp.ecdf(ti,observed,pos="median",plot=False)

fig, ax = plt.subplots()
ax.step(t1,Fhat1,where="post",label="Midpoint")
ax.step(t2,Fhat2,'--',where="post",label="Mean")
ax.step(t3,Fhat3,':',where="post",label="Median")
ax.set_xlabel("Time")
ax.set_ylabel("$\hat{F}$")    
ax.legend()

# Example 5
First, we compute the Kaplan Meier empirical CDF. 

In [ ]:
ti = [251.3, 133.3, 139.9, 261.3, 261.3, 181.9, 41.0, 228.8, 158.4, 261.3]
ti = np.array(ti)
observed = np.ones(ti.shape)
observed[ti==261.3] = 0 # censor#ed
tkm,Fhat,LCI,UCI,figKM,axKM = rpp.kaplan_meier(ti,observed,plot=True,confidence_interval="greenwood")

# add midpoint to the plot
tkm2,Fhat2 = rpp.ecdf(ti,observed,plot=False)
axKM.step(tkm2,Fhat2,linestyle='--',color="green",where="post",label="Midpoint")
plt.legend()


# Example 6
Now, we fit a Weibull using MLE. Note that the weibull_fit fuction is a custom function I'm using because the scipy.stats fitting functions don't seem to support censoring:

In [ ]:
pdhat = rd.weibull()
wb_fit = pdhat.fit(ti,[np.mean(ti),1],observed=observed)
eta_hat, beta_hat = wb_fit.params
pdhat = pdhat(beta_hat,scale=eta_hat) # Call distribution with parameters to "freeze" it
print(wb_fit.summary())

And we plot the results

In [ ]:
t = np.linspace(0,np.max(ti),1000)
figKM, axKM = plt.subplots()
axKM.step(tkm,Fhat,linestyle='-',color="blue",where="post",label="Midpoint")
axKM.fill_between(tkm,LCI,y2=UCI,linestyle=':',color="blue",step="post",label="95% CI",alpha=0.1)
axKM.plot(t,pdhat.cdf(t),linestyle='-',color='green',label="Weibull MLE")
axKM.set_xlabel("Time")
axKM.set_ylabel(r"$\hat{F}$")    
axKM.legend(loc='upper left')

# Example 7
Use graphical fitting techniques to obtain Weibull parameter estimates

In [ ]:
tmid,Fhatmid = rpp.ecdf(ti,observed,plot=False) # Using midpiont ECDF from above
y = np.log(-np.log(1-Fhatmid[1::])) # omit first point (zero). 
x = np.log(tmid[1::])

We now fit a line to this data to obtain the estimates for $\eta$ and $\beta$

In [ ]:
slope, intercept, r_value, p_value, std_err = sps.linregress(x,y)
beta = slope
eta = np.exp(-intercept/beta)
dist = rd.weibull()
dist = dist(beta,scale=eta)

In [ ]:
data = {'times':tmid[1::],'ecdf':Fhatmid[1::]}
rpp.weibull_probability_plot(dist,data=data,figsize=(7,7))

Now we add this to the empirical plot from before:

In [ ]:
figKM, axKM = plt.subplots()
axKM.step(tkm,Fhat,linestyle='-',color="blue",where="post",label="Midpoint")
axKM.fill_between(tkm,LCI,y2=UCI,linestyle=':',color="blue",step="post",label="95% CI",alpha=0.1)
axKM.plot(t,pdhat.cdf(t),linestyle='-',color='green',label="Weibull MLE")
axKM.set_xlabel("Time")
axKM.set_ylabel("$\hat{F}$")    
axKM.plot(t,dist.cdf(t),linestyle='--',color='green',label="Weibull Graphical")
axKM.legend(loc='upper left')

We'll repeat the above with MLE estimates and confidence intervals estimated from Fisher Information and the Delta method:

In [ ]:
ax = rpp.weibull_probability_plot(pdhat,data=data,confidence_bounds="time",parameter_covariance=wb_fit.cov)

In [ ]:
figKM2, axKM2 = plt.subplots()
axKM2.step(tkm,Fhat,linestyle='-',color="blue",where="post",label="Midpoint")
axKM2.fill_between(tkm,LCI,y2=UCI,linestyle=':',color="blue",step="post",label="95% CI",alpha=0.1)
axKM2.plot(t,pdhat.cdf(t),linestyle='-',color='green',label="Weibull MLE")
axKM2.set_xlabel("Time")
axKM2.set_ylabel(r"$\hat{F}$")

# compute and plot fit confidence intervals
RL,RU = rpp.weibull_reliability_confidence_interval(pdhat,t,wb_fit.cov,kind='Reliability',alpha=0.05)
axKM2.fill_between(t,1-RU,1-RL,linestyle='--',color='green',label="MLE Fit CI",alpha=0.1)

axKM2.legend(loc='upper left')

# Example 8: Example from Crow's Classic Paper
In this example, Crow simulated age data for three assets ($M=3$) with $\gamma=0.6$ and $\beta=0.5$. Note that alternatively, this could be viewed as age data coming from a single asset which has been repaired perfectly three times. The below code performs MLE on this data. Note that Crow's data is time censored at $T=T_1=T_2=T_3=200$.

In [ ]:
event_times =   [   [4.3,4.4,10.2,23.5,23.8,26.4,74.0,77.1,92.1,197.2],
                    [0.1,5.6,18.6,19.5,24.2,26.7,45.1,45.8,75.7,79.7,98.6,120.1,161.8,180.6,190.8],\
                    [8.4,32.5,44.7,48.4,50.6,73.6,98.7,112.2,129.8,136.0,195.8]
                ]
censoring_times = [200]*3
plnhpp = power_law_nhpp(1,1)
plnhpp_fit = plnhpp.fit(event_times,truncation_times=censoring_times)
print(plnhpp_fit.summary())

# Example 9: Plotting the Emperical MCF
Example from Tobias and Trindale, Chapter 12

In [ ]:
failures = [    [222,584,985,1161],\
                [273,766,1054],\
                [125,323],\
                [63,195,325],\
                [91,427,761,1096,1796] ]
suspension_times = [1901,1316,442,636,2214]
t,M,ML,MU,f,a = rpp.empirical_mean_cumulative_function(failures,suspension_times,confidence_interval="logit")
crow = power_law_nhpp(1,1)
crow_fit = crow.fit(failures,truncation_times=suspension_times)
crow.parameters = crow_fit.params

t = np.linspace(t[0],t[-1],100)
ML_fit,MU_fit = crow.mcf_confidence_interval(t,crow_fit.cov,kind="time")
a.plot(t,crow.cumulative_intensity(t),label='PL-NHPP Fit',color='red',linestyle='--')
a.fill_between(t,ML_fit,MU_fit,alpha=0.1,color='red')
plt.legend()

# Example 10: Emperical MCF
Consider the five identical asset example from http://reliawiki.org/index.php/Recurrent_Event_Data_Analysis:

| Asset | Failure Times |
|-------|---------------|
|1      | 5,10,15,17+   |
|2      | 6,13,17,19+   |
|3      | 12,20,25,26+  |
|4      | 13, 15, 24+   |
|5      | 16,22,25,28+  |

where the + indicates censoring

In [ ]:
failures = [    [5,10,15],\
                [6,13,17],\
                [12,20,25],\
                [13,15],\
                [16,22,25] ]
suspension_times = [17,19,26,24,28]
t,M,ML,MU,f,a = rpp.empirical_mean_cumulative_function( failures,
                                                        suspension_times=suspension_times,
                                                        confidence_interval="logit")
crow = power_law_nhpp(1,1)
crow_fit = crow.fit(failures,truncation_times=suspension_times)
crow.parameters = crow_fit.params

t = np.linspace(0,t[-1],100)
ML,MU = crow.mcf_confidence_interval(t,crow_fit.cov,kind="mcf")
M = crow.cumulative_intensity(t)
a.plot(t,M,label='PL-NHPP Fit',color='red',linestyle='--')
a.fill_between(t,ML,MU,alpha=0.1,color='red')
plt.legend()

# Example 11: A PL-NHPP Synthetic Example

Generate synthetic data, estimate parameters back

In [ ]:
crow_synthetic = power_law_nhpp(0.1,1.5)
M = 10
T = [5,15,10,8,7]*2 # suspension time
t_grid = np.linspace(1e-6*max(T),max(T),100)
synthetic_data = crow_synthetic.random_arrival_times(T,size=M)
t,M,ML,MU,fig,ax = rpp.empirical_mean_cumulative_function( synthetic_data,
                                                        suspension_times=T,
                                                        confidence_interval="logit",
                                                        plot=True)
synth_fit = crow_synthetic.fit(synthetic_data,truncation_times=T)
crow_synthetic.parameters = synth_fit.params
ML_fit,MU_fit = crow_synthetic.mcf_confidence_interval(t_grid,synth_fit.cov,ndt_kwds={'method':'complex','order':4})
print(synth_fit.summary())

ax.plot(t_grid,crow_synthetic.cumulative_intensity(t_grid),color='red',label="Fit")
ax.fill_between(t_grid,ML_fit,MU_fit,color='red',alpha=0.1)
ax.legend()

In [ ]:
crow_synthetic.intensity(0)

# Simulate random counts

In [ ]:
tg = np.linspace(5,max(T),10)
synthetic_counts = crow_synthetic.random_counts(tg,s=tg[0],size=100)
plt.step(tg,synthetic_counts.T,color='blue',alpha=0.1)
plt.plot(tg,synthetic_counts.mean(axis=0),color='blue',linewidth=3,label='Mean of simulations')
plt.plot(tg,crow_synthetic.cumulative_intensity(tg,t0=tg[0]),'--',color='red',label="True")
plt.legend()
plt.xlabel('t')
plt.ylabel(f'N({tg[0]},t)')


# Example 12: A system with imperfect overhauls
From Shin et al. 

In [ ]:
failures = [[116,151,213,386,387,395,407,463,492,494,501,537,564,590,609]]
pm_times = [[154, 263, 512]]
truncation_times = [612]
t,M,ML,MU,f,a = rpp.empirical_mean_cumulative_function(failures,truncation_times,confidence_interval=None)

shin = rim.imperfect_pm_minimal_cm(1,1,0.5)
shin_fit = shin.fit(failures,pm_times,truncation_times,ndt_kwds={'step':1e-5})
shin.set_parameters(*shin_fit.params)
print(shin_fit.summary())

tg = np.linspace(t[0],t[-1],100)
a.plot(tg,shin.cumulative_intensity(tg,pm_times[0],truncation_times[0]),label='Fitted M(t)')
a.legend()

# Wiener Process Model
The below code generates a Weiner Process model (with drift) and estimates the parameters back. First the simulation:

In [ ]:
mu = 1.0
sigma = 2.0
x0 = 0.0
M = 10
model = rw.Wiener(mu=mu,sigma=sigma)
times = np.linspace(0,5,10)
X = model.simulate(times,initial=x0,num_samples=M)


m = x0 + model.mu*times
U = m+1.96*sigma*np.sqrt(times)
L = m-1.96*sigma*np.sqrt(times)
plt.plot(times,X.T,color='blue',alpha=0.1)
plt.plot(times,L,color='b',ls='--')
plt.fill_between(times,L,U,alpha=0.1,color='blue')
plt.plot(times,U,color='b',ls='--')
plt.xlabel('t')
plt.ylabel('X(t)')
plt.title("Wiener Process (with drift)")

Now for the estimation:

In [ ]:
ti = [times.tolist() for ii in range(M)]
xi = X.tolist()
wiener_fit = model.fit(ti,xi,inplace=True)
print(wiener_fit.summary())

# Reflecting Brownian Motion
The below code generates a Reflecting Brownian Motion process (with drift) and estimates the parameters back. First the simulation:

In [ ]:
mu = 1.0
sigma = 2.0
x0 = 0.0
M = 10
model = rw.RBM(mu=mu,sigma=sigma)
h = 1e-1
T = 10
X = model.simulate(h,T,initial=x0,num_samples=M)

t = np.arange(0,T+h,h)
step = 1
t2 = t[step:-1:step]
# m = x0 + model.mu*t
L,U = model.get_upper_lower(t2,x0,alpha=0.05)
plt.plot(t,X.T)
# plt.plot(t,m,'k')
plt.plot(t2,L,color='b',ls='--')
plt.fill_between(t2,L,U,alpha=0.1,color='blue')
plt.plot(t2,U,color='b',ls='--')
plt.xlabel('t')
plt.ylabel('X(t)')
plt.title("Reflecting Brownian Motion")

Now for the estimation. This one takes longer because the pdf is more difficult to compute:

In [ ]:
ti = [t.tolist() for ii in range(M)]
xi = X.tolist()
# p0 is the starting point for (mu, sigma). It used to be called x0, which
# collided with x0 meaning the initial state of the process.
rbm_fit = model.fit(ti,xi,p0=[0,1])
print(rbm_fit.summary())